# Task 2 — Notebook 3: Train / Validation / Test Split

## Notebook Objective

* Split the labeled data into **Train, Validation, and Test sets before performing deep EDA**, to prevent information from the test set from influencing our decisions.
* Decide whether to use a **random split or a time-based split**, and explain the reasoning.
* Perform basic analysis to check the **date range** and **label distribution** in each split.
* Build the final **artifacts**: `train`, `validation`, and `test` files.
* Use `artifacts/labeled_table.parquet` from Notebook 2 as the input dataset.


In [1]:
import pandas as pd

labeled_table = pd.read_parquet("artifacts/labeled_table.parquet")
print(labeled_table.shape)

(96470, 21)


## 1. Basic Analysis: Random vs. Time-Based Split

Before splitting the data, we examine the date range and order volume over time. This helps us identify any trends, seasonality, or changes in the dataset and decide whether a random split or a time-based split is more appropriate.


In [3]:
print("Earliest order:", labeled_table["order_purchase_timestamp"].min())
print("Latest order:", labeled_table["order_purchase_timestamp"].max())
print("Date range span:", labeled_table["order_purchase_timestamp"].max() - labeled_table["order_purchase_timestamp"].min())

# orders per month, to see volume trend over time
orders_per_month = (
    labeled_table
    .set_index("order_purchase_timestamp")
    .resample("M")
    .size()
)
print("\nOrders per month:")
print(orders_per_month)

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37
Date range span: 713 days 02:43:59

Orders per month:
order_purchase_timestamp
2016-09-30       1
2016-10-31     265
2016-11-30       0
2016-12-31       1
2017-01-31     750
2017-02-28    1653
2017-03-31    2546
2017-04-30    2303
2017-05-31    3545
2017-06-30    3135
2017-07-31    3872
2017-08-31    4193
2017-09-30    4150
2017-10-31    4478
2017-11-30    7288
2017-12-31    5513
2018-01-31    7069
2018-02-28    6555
2018-03-31    7003
2018-04-30    6798
2018-05-31    6749
2018-06-30    6096
2018-07-31    6156
2018-08-31    6351
Freq: M, dtype: int64


## 2. Split Strategy: Time-Based Split

We chose a **time-based split** instead of a random split because it better represents real-world production.

* The model should learn from **past orders** and predict **future orders**.
* A random split could mix past and future orders between Train and Test, leading to an overly optimistic evaluation.
* The dataset shows changes in order volume over time, which may indicate **time-related patterns or data drift**.
* The time-based split keeps the Test set as future data that the model has not seen.

**Decision:** Use a **time-based split** to make the evaluation more realistic and reproducible.


## 3. Time-Based Split

We sort the orders by `order_purchase_timestamp` and split the data chronologically:

* **70% Train** → Oldest orders
* **15% Validation** → More recent orders
* **15% Test** → Newest orders

This ensures that the model learns from past data and is evaluated on future data.


In [4]:
labeled_table_sorted = labeled_table.sort_values("order_purchase_timestamp").reset_index(drop=True)

n = len(labeled_table_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = labeled_table_sorted.iloc[:train_end].copy()
val_df = labeled_table_sorted.iloc[train_end:val_end].copy()
test_df = labeled_table_sorted.iloc[val_end:].copy()

print("Train:", train_df.shape, train_df["order_purchase_timestamp"].min(), "->", train_df["order_purchase_timestamp"].max())
print("Val:  ", val_df.shape, val_df["order_purchase_timestamp"].min(), "->", val_df["order_purchase_timestamp"].max())
print("Test: ", test_df.shape, test_df["order_purchase_timestamp"].min(), "->", test_df["order_purchase_timestamp"].max())

Train: (67529, 21) 2016-09-15 12:16:38 -> 2018-04-15 20:12:35
Val:   (14470, 21) 2018-04-15 20:17:11 -> 2018-06-21 08:29:29
Test:  (14471, 21) 2018-06-21 08:41:07 -> 2018-08-29 15:00:37


## 4. Check Date Range and Label Balance

After the time-based split, we verify that:

* Each dataset has a logical and continuous **date range**.
* There is **no overlap** between Train, Validation, and Test periods.
* We check the percentage of `is_late = 1` in each split.
* The percentages may differ slightly because the split is **not stratified**, but they should remain reasonably close.

This confirms that the time-based split was performed correctly and that the label distribution is not significantly different across the three datasets.


In [5]:
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    pct_late = df["is_late"].mean() * 100
    print(f"{name:6s} | rows={df.shape[0]:6d} | is_late rate={pct_late:.2f}%")

Train  | rows= 67529 | is_late rate=9.03%
Val    | rows= 14470 | is_late rate=5.34%
Test   | rows= 14471 | is_late rate=6.61%


### Observations

* **Train:** `is_late = 9.03%`
* **Validation:** `is_late = 5.34%`
* **Test:** `is_late = 6.61%`

The late-order rates are **not closely balanced** across the three splits. The Train set has a noticeably higher late-order rate than Validation and Test.

This difference may be caused by **changes in delivery performance over time**. Since we used a time-based split, this could represent a real temporal pattern in the data rather than a splitting error.

This pattern should be investigated further during **EDA in Notebook 4**.


## 5. Sanity Checks

Before saving the splits, we verify that:

1. The total number of rows in **Train + Validation + Test** equals the original `labeled_table` row count.
2. There are **no shared `order_id` values** between Train, Validation, and Test, preventing data leakage.
3. The splits are correctly ordered by time:

   * Train ends before or at the start of Validation.
   * Validation ends before or at the start of Test.

These checks confirm that the time-based split was performed correctly without losing, duplicating, or leaking data.


In [6]:
assert train_df.shape[0] + val_df.shape[0] + test_df.shape[0] == labeled_table.shape[0], \
    "Split sizes don't add up to the original table!"

train_ids = set(train_df["order_id"])
val_ids = set(val_df["order_id"])
test_ids = set(test_df["order_id"])

assert len(train_ids & val_ids) == 0, "Overlap between train and val!"
assert len(train_ids & test_ids) == 0, "Overlap between train and test!"
assert len(val_ids & test_ids) == 0, "Overlap between val and test!"

assert train_df["order_purchase_timestamp"].max() <= val_df["order_purchase_timestamp"].min(), \
    "Train dates overlap with val dates!"
assert val_df["order_purchase_timestamp"].max() <= test_df["order_purchase_timestamp"].min(), \
    "Val dates overlap with test dates!"

print("All sanity checks passed.")
print(f"Train: {train_df.shape[0]} ({train_df.shape[0]/n*100:.1f}%)")
print(f"Val:   {val_df.shape[0]} ({val_df.shape[0]/n*100:.1f}%)")
print(f"Test:  {test_df.shape[0]} ({test_df.shape[0]/n*100:.1f}%)")

All sanity checks passed.
Train: 67529 (70.0%)
Val:   14470 (15.0%)
Test:  14471 (15.0%)


## 6. Save the Artifacts

We save the three datasets as separate files:

* `train.parquet`
* `validation.parquet`
* `test.parquet`

Notebook 4 (EDA) will use **only `train.parquet`** and will not access the Test set, preventing information leakage from the Test data.


In [7]:
import os

os.makedirs("artifacts", exist_ok=True)

train_df.to_parquet("artifacts/train.parquet", index=False)
val_df.to_parquet("artifacts/validation.parquet", index=False)
test_df.to_parquet("artifacts/test.parquet", index=False)

print("Saved artifacts/train.parquet ->", train_df.shape)
print("Saved artifacts/validation.parquet ->", val_df.shape)
print("Saved artifacts/test.parquet ->", test_df.shape)

Saved artifacts/train.parquet -> (67529, 21)
Saved artifacts/validation.parquet -> (14470, 21)
Saved artifacts/test.parquet -> (14471, 21)
